# Topic Models: Latent Dirichlet Allocation

The narrative companion to formalNLP's twenty-fifth topic. Every number here comes from `topic_models_lda.py`, whose
harness asserts each claim the page makes. Two testbeds carry the topic: small simulated LDA worlds, where the topics
are known exactly, and the shared finance corpus, whose generator draws every document's content nouns half from its
sector's vocabulary and half from a shared one. Run from the repository root.

In [ ]:
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path("notebooks/topic-models-lda").resolve()))
import topic_models_lda as lda

np.set_printoptions(precision=4, suppress=True)
fd = lda.fin_data("restaurant")
print("finance restaurant vocabulary:", fd["V"], "words,", int(fd["X"].sum()), "tokens in", len(fd["docs"]), "documents")

## 1. The generative model (Def 1, Def 2, Prop 1, Prop 2, Rem 1)

LDA draws topics $\varphi_k \sim \mathrm{Dir}(\beta\mathbf{1})$, proportions $\theta_d \sim \mathrm{Dir}(\alpha)$, and each token's
topic and word. Given $\theta_d$ the words are i.i.d. from $p_d = \Phi^\top\theta_d$, a point of the topic polytope
$\operatorname{conv}\{\varphi_k\}$. A document's observed frequencies $\hat p_d$ scatter around $p_d$, often outside the
polytope; fitting $\theta_d$ by maximum likelihood is the KL projection of $\hat p_d$ back onto it.

In [ ]:
lda.test_prop1_documents_live_in_the_polytope()
lda.test_prop2_fitting_theta_is_a_kl_projection()
for a in lda.SIMPLEX_ALPHAS:
    print(f"alpha {a:5}: share of p-hat outside the topic triangle at n_d = 10/40/160:",
          [lda.simplex_world(a, n)["outside"] for n in lda.SIMPLEX_LENGTHS])

Latent semantic analysis projects in Euclidean norm onto a subspace whose coordinates take both signs (Rem 1): its
rank-6 reconstruction of the finance documents' word frequencies has negative entries, which no mixture of
distributions can produce.

In [ ]:
lda.test_rem1_lsa_coordinates_are_signed()
X = fd["X"]; P = X / X.sum(1, keepdims=True)
U, sv, Vt = np.linalg.svd(P, full_matrices=False)
print("share of negative entries in LSA's rank-6 reconstruction:", round(float(((U[:, :6] * sv[:6]) @ Vt[:6] < -1e-12).mean()), 6))

## 2. Identifiability and anchor words (Prop 3, Prop 4, Def 3, Prop 5, Thm 1, Algorithm 1)

Topics are identified only up to relabeling, and without further conditions the polytope can be stretched (Prop 3).
A topic that every document carries at one weight cannot be separated from the others at all (Prop 4). Under
separability the per-document co-occurrence estimator is unbiased (Prop 5), every row of $\bar Q$ is a convex
combination of the anchor rows, and with a nonsingular $R$ the anchors are exactly the vertices (Thm 1).

In [ ]:
for t in (lda.test_prop3_relabeling_and_stretching, lda.test_prop4_shared_topic_is_unidentifiable,
          lda.test_prop5_cooccurrence_estimator_is_unbiased, lda.test_thm1_anchor_rows_are_the_vertices,
          lda.test_thm1_needs_nonsingular_r):
    t()
W = lda.thm1_world(singular=True)
print("singular R: the shared topic's anchor row lies in the others' hull, LP distance",
      round(lda.hull_distance(W["Qbar"][3], W["Qbar"][[0, 1, 2]]), 12))
print("FastAnchorWords picks", sorted(lda.fast_anchor_words(W["Qbar"], 4, np.arange(W["V"]))), "instead of [0, 1, 2, 3]")

Algorithm 1 on sampled documents. With dense topics, only the planted anchors can be vertices; without them, the
recovery error stops at a floor that no number of documents removes.

In [ ]:
lda.test_anchor_recovery_improves_with_documents()
for sep in (True, False):
    print("separable  " if sep else "no anchors ", [round(float(lda.anchor_recovery(sep, M)["l1"].mean()), 6) for M in lda.ANCHOR_M],
          "population", round(float(lda.anchor_recovery(sep, None)["l1"].mean()), 6))

## 3. Collapsed Gibbs sampling (Lemma 1, Thm 2, Algorithm 2, Cor 1, Prop 6)

Integrating $\theta$ and $\Phi$ out leaves a full conditional in three counts. On a seven-token corpus we can write
down the whole Markov chain: one systematic sweep leaves the enumerated posterior exactly invariant.

In [ ]:
for t in (lda.test_lemma1_dirichlet_multinomial_marginal, lda.test_thm2_full_conditional,
          lda.test_cor1_the_posterior_is_stationary, lda.test_algorithm2_sampler_visits_the_posterior,
          lda.test_prop6_point_estimates_are_posterior_means):
    t()
states, post, log_pw = lda.toy_posterior()
print(len(states), "states; log p(w) =", round(log_pw, 6), "; most probable state", states[int(post.argmax())], round(float(post.max()), 6))

## 4. Mean-field variational inference (Lemma 2, Thm 3, Cor 2)

$\mathbb{E}[\log\theta_k] = \psi(\gamma_k) - \psi(\sum_j\gamma_j)$ gives LDA's coordinate updates. Every block update raises
the ELBO, and our bound equals scikit-learn's to $10^{-8}$ relative.

In [ ]:
for t in (lda.test_lemma2_dirichlet_log_moment, lda.test_cor2_elbo_never_decreases, lda.test_thm3_bound_matches_scikit_learn):
    t()
lda.test_the_prior_ranks_the_engines()
for b in lda.GV_BETAS:
    g = lda.gibbs_vs_vi(b)
    print(f"beta {b:5}: mean L1  Gibbs {g['gibbs_l1'].mean():.6f}  CAVI {g['cavi_l1'].mean():.6f}  anchors {g['anchors_l1'].mean():.6f}")

## 5. Evaluating a topic model (Def 4, Thm 4, Prop 7, Def 5)

The harmonic mean of posterior likelihoods is consistent for $p(w)$ but biased upward, and with a continuous $\theta$
its variance is infinite as soon as a document holds $\alpha_k$ or more tokens of topic $k$'s anchor words. On a finite
collapsed state space its variance is finite, and on the toy smaller than prior importance sampling's.

In [ ]:
for t in (lda.test_thm4_harmonic_mean_identity_and_bias, lda.test_thm4c_anchor_tokens_make_the_variance_infinite,
          lda.test_coherence_definitions):
    t()
for n in (4, 8, 12):
    m = lda.toy_marginals([[i % 3 for i in range(n)]], 3, 2, 0.5, 0.5)
    print(f"n = {n:2d}: relative variance  harmonic mean {lda.hm_relative_variance(m):.4f}   prior IS {lda.is_relative_variance(m):.4f}")

## 6. Finance case study: seven planted vocabularies, six identifiable topics

The generator seats content nouns in six sector vocabularies and one shared vocabulary, and every document draws
about half its nouns from each of two: its sector's and the shared one. That is Prop 4: the identifiable truth is
six topics $\psi_s$, and the anchor search finds one anchor per sector and none for the shared vocabulary.

In [ ]:
for t in (lda.test_finance_text_is_topic6s_and_continues_it, lda.test_seven_restaurants_every_document_half_and_half,
          lda.test_finance_anchors_one_per_sector_none_shared):
    t()
fa = lda.fin_anchors()
print("median shared-vocabulary share:", round(float(np.median(fd["theta7"][:, 6])), 6))
print("six anchors:", [(fd["vocab"][a], lda.TOPICS[fd["home"][a]]) for a in fa["a6"]])
print("seventh pick:", fd["vocab"][fa["a7"][-1]], "->", lda.TOPICS[fd["home"][fa["a7"][-1]]])

The engines on the restaurant vocabulary ($K = 6$, three seeds). Collapsed Gibbs finds the sectors; CAVI from
noise stalls where its own bound is far lower than at the sectors, and EM for the single-membership model, the
correctly specified model here, stalls likewise. The obstacle is the optimizer, not the model.

In [ ]:
for t in (lda.test_mou_oracle_beats_lda_but_em_cannot_find_it, lda.test_cavi_stalls_where_its_own_bound_is_lower,
          lda.test_scikit_learn_also_stops_short):
    t()
for name, row in lda.fin_table().items():
    print(f"{name:11} L1 {np.mean(row['l1']):.6f}  completion {np.mean(row['completion']):.6f}  NMI {np.mean(row['nmi']):.6f}")
print("anchors     L1", round(float(fa["l1"].mean()), 6))
print("scikit-learn batch VB:", {k: round(v, 6) for k, v in lda.fin_sklearn().items()})

With every word kept, one topic absorbs the function words and the rest key on company names, which carry the
sector. Tone and document type never surface. `topic_mixtures` is the stage the capstone imports.

In [ ]:
lda.test_templates_hide_tone_and_document_type()
for name, v in lda.fin_templates().items():
    print(name, {k: round(v[k], 6) for k in ("sector", "doc_type", "tone", "max_theta")})
    print("   ", v["top_words"][0])
lda.test_topic_mixtures_reads_held_out_sectors()
theta, labels = lda.topic_mixtures(lda.finance_run()["held"][:3])
print(labels)
print(theta.round(3))

## The harness

Every claim above, run in one pass (about 45 seconds).

In [ ]:
assert lda._run_harness() == 0